In [13]:
## Imports

import sys, os
from pathlib import Path

parent_folder = str(Path.cwd().parents[3])
if parent_folder not in sys.path:
    sys.path.append(parent_folder)

from sigpy import mri
import scipy
import pickle
import sigpy as sp
import cupy as cp
import numpy as np
from scipy.io import savemat, loadmat
import twixtools
import matplotlib.pyplot as plt


## My files
import save_data_helpers
import recon_plot_helpers
from custom_recons.created_app import TotalVariationRecon_Stacked

In [14]:
data_bins = save_data_helpers.read_pickle('/data/lilianae/data_and_spoke_bins/data_bins_100sp_phil_gating.pkl')
dcf_bins = save_data_helpers.read_pickle('/data/lilianae/recons_pkl_gated/dcf_ksp_100sp_phil_gating_512_5gates.pkl')
spoke_bins = save_data_helpers.read_pickle('/data/lilianae/data_and_spoke_bins/spoke_bins_100sp_phil_gating.pkl')
mps = save_data_helpers.read_pickle('/home/lilianae/projects/naf_clean/coils/subject2_mid0082/espirit_mps_full_res_ksp_512_ungated.pkl')


num_gates = len(data_bins)
data_bins_with_dcf = [None] * num_gates

## Create list of data_bins with dcf applied
for gate in range(num_gates):
    data_bins_with_dcf[gate] = data_bins[gate] * dcf_bins[gate]
    print(f'Data bins w/ dcf shape = {data_bins_with_dcf[gate].shape}')
    print(f'coords shape = {spoke_bins[gate].shape}')


Data bins w/ dcf shape = (15, 58, 100, 512)
coords shape = (58, 100, 512, 3)
Data bins w/ dcf shape = (15, 58, 100, 512)
coords shape = (58, 100, 512, 3)
Data bins w/ dcf shape = (15, 58, 100, 512)
coords shape = (58, 100, 512, 3)
Data bins w/ dcf shape = (15, 58, 100, 512)
coords shape = (58, 100, 512, 3)
Data bins w/ dcf shape = (15, 58, 100, 512)
coords shape = (58, 100, 512, 3)


In [12]:
def _stacked_nufft_operator_sens(img_shape, coords, mps):
    """setup a stacked 2D NUFFT sp operator acting on a 3D image
       the opeator first performs a 1D FFT along the "z" axis (0 or left-most axis)
       followed by applying 2D NUFFTS to all "slices"
       
    Parameters
    ----------
        img_shape: tuple
            shape of the image
        coords: (numpy or cupy) array 
            coordinates of the k-space samples
            shape (n_k_space_points,2)
            units: "unitless" -> -N/2 ... N/2 at Nyquist (sp convention)
        mps: (numpy or cupy) array
            sensitivity maps of shape (num_channels, *img_shape)

    Returns
    -------
        Diag: a stack of NUFFT operators
    """

    num_channels = len(mps)

    ft0_op = sp.linop.FFT(img_shape, axes=(0, ))

    # setup a 2D NUFFT operator for the start
    nufft_op = sp.linop.NUFFT(img_shape[1:], coords)


    # reshaping operator for input
    rs_in = sp.linop.Reshape(img_shape[1:], (1, ) + img_shape[1:])
    # setup a list of "n" 2D NUFFT operators (one per slice)
    ops = []
    for i in range(img_shape[0]):
        coords_i = coords[i].reshape(-1, coords.shape[-1])[:, 1:]  # (400*512, 2)
        nufft_op_i = sp.linop.NUFFT(img_shape[1:], coords_i)
        # Reshape NUFFT output from flat to 2D: (400*512,) -> (400, 512)
        rs_nufft = sp.linop.Reshape((coords.shape[1], coords.shape[2]), nufft_op_i.oshape)
        rs_out_i = sp.linop.Reshape((1, coords.shape[1], coords.shape[2]), (coords.shape[1], coords.shape[2]))
        ops.append(rs_out_i * rs_nufft * nufft_op_i * rs_in)


    # apply 2D NUFFTs to all "slices" using the sp Diag operator
    full_op= sp.linop.Diag(ops, iaxis=0, oaxis=0) * ft0_op
    #### Combine Sensitivity Op (mult with sens) and respective ft0+nuFFT op:

    #sensitivity = np.ones((num_channels,*img_shape),dtype=np.complex64)
    S = sp.linop.Multiply(img_shape,mps)

    rs_in_sense = sp.linop.Reshape(img_shape,(1,)+img_shape)
    rs_out_sense = sp.linop.Reshape((1,)+tuple(full_op.oshape),full_op.oshape)
    return  sp.linop.Diag(num_channels*[rs_out_sense*full_op*rs_in_sense],iaxis=0,oaxis=0)*S

In [ ]:
img_shape = (58, 512, 512)
device = 2

with cp.cuda.Device(device):
    ###################################################################
    # PRE-INITIALIZATION: Independent TV reconstructions
    ###################################################################

    ind_recons = cp.zeros((num_gates, *img_shape), dtype=cp.complex64)

    for i in range(num_gates):
        print(f"\rPre-initialization: TV recon for gate {i}/{num_gates}", end='', flush=True)
        print()
        tv_preinit_alg = TotalVariationRecon_Stacked(y=data_bins_with_dcf,
                                            mps=mps,
                                            lamda=1e-3, 
                                            coord=spoke_bins[i],
                                            device=device,
                                            z=None, 
                                            max_iter=30,
                                            max_power_iter=10,
                                            show_pbar=True)
        ind_recons[i, ...] = tv_preinit_alg.run()
        del tv_preinit_alg